# Telecom Churn Prediction - Exploratory Data Analysis

This notebook performs exploratory data analysis on the telecom churn dataset.

## 1. Import Libraries

In [ ]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.join(os.path.abspath('..'), 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from visualization import ChurnVisualizer

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 2. Load Data

In [ ]:
# Load the dataset
DATA_PATH = '../data/raw/telecom_churn_data.csv'

# Generate sample data if not exists
if not os.path.exists(DATA_PATH):
    print("Data file not found. Generating sample data...")
    from data_generator import generate_sample_telecom_data
    df = generate_sample_telecom_data(n_samples=5000, output_path=DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
df.head()

## 3. Dataset Overview

In [ ]:
# Display basic information
print("Dataset Info:")
print("=" * 50)
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary:")
print("=" * 50)
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print("=" * 50)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
missing_df[missing_df['Missing'] > 0]

## 4. Churn Distribution Analysis

In [ ]:
# Churn distribution
print("Churn Distribution:")
print("=" * 50)
print(df['Churn'].value_counts())
print(f"\nChurn Rate: {(df['Churn'].value_counts()['Yes'] / len(df) * 100):.2f}%")

# Visualize churn distribution
visualizer = ChurnVisualizer()
visualizer.plot_churn_distribution(df, target_col='Churn')

## 5. Numerical Features Analysis

In [ ]:
# Numerical features
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(numerical_features):
    axes[idx].hist(df[feature], bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_xlabel(feature, fontsize=12)
    axes[idx].set_ylabel('Frequency', fontsize=12)
    axes[idx].set_title(f'Distribution of {feature}', fontsize=14, fontweight='bold')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Numerical features by churn
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(numerical_features):
    df[df['Churn'] == 'No'][feature].hist(ax=axes[idx], bins=30, alpha=0.5, label='No Churn', edgecolor='black')
    df[df['Churn'] == 'Yes'][feature].hist(ax=axes[idx], bins=30, alpha=0.5, label='Churn', edgecolor='black')
    axes[idx].set_xlabel(feature, fontsize=12)
    axes[idx].set_ylabel('Frequency', fontsize=12)
    axes[idx].set_title(f'{feature} by Churn Status', fontsize=14, fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Categorical Features Analysis

In [ ]:
# Key categorical features
categorical_features = ['Contract', 'InternetService', 'PaymentMethod', 'gender']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, feature in enumerate(categorical_features):
    churn_counts = df.groupby([feature, 'Churn']).size().unstack(fill_value=0)
    churn_counts.plot(kind='bar', ax=axes[idx], stacked=False, color=['#66b3ff', '#ff6666'])
    axes[idx].set_xlabel(feature, fontsize=12)
    axes[idx].set_ylabel('Count', fontsize=12)
    axes[idx].set_title(f'{feature} by Churn Status', fontsize=14, fontweight='bold')
    axes[idx].legend(['No Churn', 'Churn'])
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
# Prepare data for correlation (encode categorical variables)
df_corr = df.copy()

# Encode binary categorical variables
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}

for col in df_corr.select_dtypes(include=['object']).columns:
    if col != 'customerID':
        if df_corr[col].nunique() <= 2:
            df_corr[col] = df_corr[col].map(binary_map).fillna(df_corr[col])
        else:
            df_corr[col] = pd.Categorical(df_corr[col]).codes

# Drop customerID
if 'customerID' in df_corr.columns:
    df_corr = df_corr.drop('customerID', axis=1)

# Calculate correlation matrix
corr_matrix = df_corr.corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with Churn
churn_corr = corr_matrix['Churn'].sort_values(ascending=False)
print("Correlation with Churn:")
print("=" * 50)
print(churn_corr)

# Visualize
plt.figure(figsize=(10, 8))
churn_corr[1:].plot(kind='barh', color='teal')
plt.xlabel('Correlation Coefficient', fontsize=12)
plt.title('Feature Correlation with Churn', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Key Insights

In [ ]:
print("KEY INSIGHTS FROM EDA:")
print("=" * 80)
print("\n1. CHURN DISTRIBUTION:")
print(f"   - Total customers: {len(df)}")
print(f"   - Churned customers: {len(df[df['Churn'] == 'Yes'])}")
print(f"   - Churn rate: {(len(df[df['Churn'] == 'Yes']) / len(df) * 100):.2f}%")

print("\n2. CONTRACT TYPE IMPACT:")
for contract_type in df['Contract'].unique():
    subset = df[df['Contract'] == contract_type]
    churn_rate = (len(subset[subset['Churn'] == 'Yes']) / len(subset) * 100)
    print(f"   - {contract_type}: {churn_rate:.2f}% churn rate")

print("\n3. TENURE ANALYSIS:")
print(f"   - Average tenure (No Churn): {df[df['Churn'] == 'No']['tenure'].mean():.2f} months")
print(f"   - Average tenure (Churn): {df[df['Churn'] == 'Yes']['tenure'].mean():.2f} months")

print("\n4. MONTHLY CHARGES:")
print(f"   - Average charges (No Churn): ${df[df['Churn'] == 'No']['MonthlyCharges'].mean():.2f}")
print(f"   - Average charges (Churn): ${df[df['Churn'] == 'Yes']['MonthlyCharges'].mean():.2f}")

print("\n" + "=" * 80)

## Conclusion

This exploratory data analysis reveals several important patterns:

1. **Contract Type**: Month-to-month contracts show higher churn rates
2. **Tenure**: Customers with shorter tenure are more likely to churn
3. **Monthly Charges**: Higher monthly charges are associated with increased churn
4. **Services**: Lack of additional services (tech support, online security) correlates with higher churn

These insights will guide feature engineering and model development for churn prediction.